<a href="https://colab.research.google.com/github/sergioGarcia91/SeismicUP/blob/main/08b_DBSCAN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Inicio

In [ ]:
!git clone https://github.com/sergioGarcia91/SeismicUP.git

In [ ]:
# para incluir la libreria clonada
import sys
sys.path.append("/content/SeismicUP")

import seismicup as sup

In [ ]:
!pip -q install python-calamine

In [ ]:
!pip3 install contextily

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
import seaborn as sns
import os
import re
import contextily as cx #para el basemap en geopandas
import xyzservices.providers as xyz #para escoger el basemap
import seismicup as sup
import urllib.request
import matplotlib.font_manager as fm
import random

from matplotlib.colors import LogNorm # para la escala logaritmica de los colores
from matplotlib.ticker import LogFormatter, LogLocator # escala log
from scipy import stats # regresion lineal

# Conectar al Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Cambiar Fuente

In [ ]:
sup.plots.get_TimesNewRoman_font()

# Paths

In [ ]:
path_save_figures = ''

# Cargar CSV

In [ ]:
df_5km = pd.read_csv('/content/SeismicUP/Outputs/DBSCAN_5km_n100.csv',
                     sep=';',
                     decimal=',')

df_5km = df_5km[df_5km['Cluster'] != -1].copy()
df_5km.reset_index(drop=True, inplace=True)
df_5km['Tiempo'] = pd.to_datetime(df_5km['Fecha-Hora UTC'], dayfirst=True)
df_5km['year'] = df_5km['Tiempo'].dt.year

df_5km.head()

In [ ]:
df_7km = pd.read_csv('/content/SeismicUP/Outputs/DBSCAN_7km_n50.csv',
                     sep=';',
                     decimal=',')

df_7km = df_7km[df_7km['Cluster'] != -1].copy()
df_7km.reset_index(drop=True, inplace=True)
df_7km['Tiempo'] = pd.to_datetime(df_7km['Fecha-Hora UTC'], dayfirst=True)
df_7km['year'] = df_7km['Tiempo'].dt.year

df_7km.head()

# Centros P50
El centro de cada clúster se calculará utilizando el percentil 50 (P50) de las coordenadas X, Y y Z.

## 5 km

In [ ]:
dict_temp = {'cluster':[],
             'X [m] p50':[],
             'Y [m] p50':[],
             'Z [km] p50':[],}
for cluster in df_5km['Cluster'].unique():
  df_cluster = df_5km[df_5km['Cluster'] == cluster].copy()

  dict_temp['cluster'].append(cluster)
  dict_temp['X [m] p50'].append(df_cluster['X [m]'].quantile(0.5))
  dict_temp['Y [m] p50'].append(df_cluster['Y [m]'].quantile(0.5))
  dict_temp['Z [km] p50'].append(df_cluster['Profundidad [km]'].quantile(0.5))

df_centros_5km = pd.DataFrame(dict_temp)

df_centros_5km['n'] = 0
df_centros_5km['Cr_min'] = 0
df_centros_5km['Cr_max'] = 0
df_centros_5km['Dc'] = 0
df_centros_5km['mc'] = 0
df_centros_5km['a'] = 0
df_centros_5km['b'] = 0
df_centros_5km['mc_std'] = 0
df_centros_5km['a_std'] = 0
df_centros_5km['b_std'] = 0

df_centros_5km

## 7 km

In [ ]:
dict_temp = {'cluster':[],
             'X [m] p50':[],
             'Y [m] p50':[],
             'Z [km] p50':[],}
for cluster in df_7km['Cluster'].unique():
  df_cluster = df_7km[df_7km['Cluster'] == cluster].copy()

  dict_temp['cluster'].append(cluster)
  dict_temp['X [m] p50'].append(df_cluster['X [m]'].quantile(0.5))
  dict_temp['Y [m] p50'].append(df_cluster['Y [m]'].quantile(0.5))
  dict_temp['Z [km] p50'].append(df_cluster['Profundidad [km]'].quantile(0.5))

df_centros_7km = pd.DataFrame(dict_temp)

df_centros_7km['n'] = 0
df_centros_7km['Cr_min'] = 0
df_centros_7km['Cr_max'] = 0
df_centros_7km['Dc'] = 0
df_centros_7km['mc'] = 0
df_centros_7km['a'] = 0
df_centros_7km['b'] = 0
df_centros_7km['mc_std'] = 0
df_centros_7km['a_std'] = 0
df_centros_7km['b_std'] = 0


df_centros_7km

# Gutenberg Richter por cluster

In [ ]:
mag_min = 0
mag_max = 10
steps_mag = 0.1
porcentaje_remuestreo = 0.8
numero_remuestreos = 1000
reemplazo = False
correcion_mc = 0.0

## 5 Km

In [ ]:
for cluster in df_5km['Cluster'].unique():
  df_cluster = df_5km[df_5km['Cluster'] == cluster].copy()
  x_lim = (df_cluster['X [m]'].min(), df_cluster['X [m]'].max())
  x_lim = x_lim[0] + (x_lim[1]-x_lim[0])/2
  x_lim = (x_lim-150000, x_lim+150000)
  y_lim = (df_cluster['Y [m]'].min(), df_cluster['Y [m]'].max())
  y_lim = y_lim[0] + (y_lim[1]-y_lim[0])/2
  y_lim = (y_lim-150000, y_lim+150000)

  gdf_cluster = gpd.GeoDataFrame(
    df_cluster, geometry=gpd.points_from_xy(df_cluster.Longitud, df_cluster.Latitud), crs="EPSG:4326"
    )
  gdf_cluster = gdf_cluster.to_crs(9377)

  fig, ax = plt.subplots(figsize=(7, 7), ncols=1, nrows=1)


  gdf_cluster.plot(color='r',
                   ax=ax,
                   zorder=100,
                   label=f'Cluster {cluster} [DBSCAN 5 km]',)

  ax.set_xlim(x_lim[0] , x_lim[1])
  ax.set_ylim(y_lim[0] , y_lim[1])

  cx.add_basemap(ax=ax,
                crs='epsg:9377', # el sistema de coordenadas
                source=xyz.OpenTopoMap,
                reset_extent=True,
                zorder=0,
                zoom=8,
                attribution_size=5) # Para incluir un mapa base

  ax.set_xlabel('X [m]')
  ax.set_ylabel('Y [m]')

  ax.set_aspect('equal')
  plt.grid(True,
          ls='--',
          color='gray',
          alpha=0.8)


  legend  = plt.legend(loc='upper center')
  legend.set_zorder(100)

  sup.plots.save_plot(path_save_figuras = path_save_figures,
                    file_name = f'Map_DBSCAN_5km_cluster_{cluster}',
                    formato = 'png',
                    dpi_ = 300)

  plt.show()
  print('\n')

In [ ]:
for cluster in df_5km['Cluster'].unique():
  fig, ax = plt.subplots(figsize=(10,3), nrows=1, ncols=2)

  filtro_cluster = df_5km['Cluster'] == cluster
  ax[0].hist(df_5km['year'][filtro_cluster],
             bins=np.arange(1992, 2025, 1),
             )
  ax[0].set_xlabel('Year')
  ax[0].set_ylabel('Frequency')
  ax[0].grid(ls='--', alpha= 0.8)

  ax[1].hist(df_5km['Profundidad [km]'][filtro_cluster],
           bins=np.arange(0, 65, 2.5),
           )
  ax[1].set_xlabel('Depth [km]')
  ax[1].set_ylabel('Frequency')
  ax[1].grid(ls='--', alpha= 0.8)

  radios_ = np.logspace(0, 1.5, 50) # km
  cr = sup.stats.correlacion_integral_xyz(x_km = df_5km['X [km]'][filtro_cluster],
                                        y_km = df_5km['Y [km]'][filtro_cluster],
                                        z_km = df_5km['Profundidad [km]'][filtro_cluster],
                                        radio_en_km = radios_)

  slope, intercepto, otros = sup.stats.estimar_Dc(r_km = radios_,
                                                 C = cr,
                                                 idx_range=None)

  cr_min = radios_[otros[3]][otros[2]][0]
  cr_max = radios_[otros[3]][otros[2]][-1]

  df_centros_5km.loc[df_centros_5km['cluster'] == cluster, 'Cr_min'] = cr_min
  df_centros_5km.loc[df_centros_5km['cluster'] == cluster, 'Cr_max'] = cr_max
  df_centros_5km.loc[df_centros_5km['cluster'] == cluster, 'Dc'] = slope

  plt.suptitle(f'Cluster {cluster} [DBSCAN 5 km] – C(r): {cr_min:.2f}-{cr_max:.2f} km – Dc= {slope:.2f}')

  sup.plots.save_plot(path_save_figuras = path_save_figures,
                    file_name = f'Hist_DBSCAN_5km_cluster_{cluster}',
                    formato = 'png',
                    dpi_ = 300)

  plt.show()
  print()

In [ ]:
dicc_5km = {}

for cluster in df_5km['Cluster'].unique():
  dicc_temp ={'n':[],
              'mc':[],
            'a':[],
            'b':[],
            'mc_std':[],
            'a_std':[],
            'b_std':[],
            'get_N':[]}

  df_cluster = df_5km[df_5km['Cluster'] == cluster].copy()

  dicc_temp['Ml'] = df_cluster['Magnitud'].values

  temp = sup.stats.obtener_N_LGR(magnitudes_array = df_cluster['Magnitud'].values,
                                                  magnitud_minima = mag_min,
                                                  magnitud_maxima = mag_max,
                                                  steps_ = steps_mag,
                                                  texto = False)

  dicc_temp['get_N'].append(temp)

  temp = sup.stats.get_GutenbergRichter_values_boosted_2(magnitudes = df_cluster['Magnitud'],
                                                            percentage_data = porcentaje_remuestreo, # 0.8
                                                            magnitud_minima = mag_min,
                                                            magnitud_maxima = mag_max,
                                                            steps_ = steps_mag,
                                                            n_resamples = numero_remuestreos,
                                                            replace_ = reemplazo,
                                                            correcion_m = correcion_mc)
  dicc_temp['n'].append(df_cluster.shape[0])
  dicc_temp['mc'].append(temp['mc_mean'])
  dicc_temp['a'].append(temp['a_mean'])
  dicc_temp['b'].append(temp['b_mean'])
  dicc_temp['mc_std'].append(temp['mc_std'])
  dicc_temp['a_std'].append(temp['a_std'])
  dicc_temp['b_std'].append(temp['b_std'])

  df_centros_5km.loc[df_centros_5km['cluster'] == cluster, 'n'] = df_cluster.shape[0]
  df_centros_5km.loc[df_centros_5km['cluster'] == cluster, 'mc'] = temp['mc_mean']
  df_centros_5km.loc[df_centros_5km['cluster'] == cluster, 'a'] = temp['a_mean']
  df_centros_5km.loc[df_centros_5km['cluster'] == cluster, 'b'] = temp['b_mean']
  df_centros_5km.loc[df_centros_5km['cluster'] == cluster, 'mc_std'] = temp['mc_std']
  df_centros_5km.loc[df_centros_5km['cluster'] == cluster, 'a_std'] = temp['a_std']
  df_centros_5km.loc[df_centros_5km['cluster'] == cluster, 'b_std'] = temp['b_std']

  dicc_5km[str(cluster)] = dicc_temp

dicc_5km

In [ ]:
for cluster in dicc_5km.keys():
  print(f'\nCluster: {cluster}')
  fig, ax = plt.subplots(nrows=1, ncols=2, figsize=(8,4),
                         sharex='all')

  temp = dicc_5km[cluster]
  min_max_GR = [temp['mc'][0].round(2) - 0.2,
                max(temp['get_N'][0]['m'][temp['get_N'][0]['N'] != 0]) + 0.2]

  sup.plots.histograma_plot(df= pd.DataFrame(temp['Ml'], columns=['Ml']),
                          columna= 'Ml',
                          ax_h= ax[0],
                          dist_min= 0,
                          dist_max= 10,
                          units_label='Ml',
                          step_= 0.2)


  ax[1].axvline(x=3, c='k', ls='--', alpha=0.5)

  sup.plots.GutenbergRichter_scatter_plot(N_count = temp['get_N'][0]['N'],
                                          magnitudes = temp['get_N'][0]['m'],
                                          ax_h = ax[1],
                                          mc = temp['mc'][0].round(2),
                                          magnitud_maxima= mag_max)
  sup.plots.GutenbergRichter_line_plot(b_value = temp['b'][0],
                                        a_value = temp['a'][0],
                                        ax_h = ax[1],
                                        magnitud_minima = min_max_GR[0],
                                        magnitud_maxima = min_max_GR[1])

  ax[1].set_xlabel('Magnitude')
  ax[1].set_title(f'Gutenberg-Richter')

  plt.suptitle(f'Cluster: {cluster} [n={temp['n'][0]}] [DBSCAN 5 km]')
  plt.tight_layout()

  sup.plots.save_plot(path_save_figuras = path_save_figures,
                    file_name = f'LGR_DBSCAN_5km_cluster_{cluster}',
                    formato = 'png',
                    dpi_ = 300)

  plt.show()


## 7 Km

In [ ]:
for cluster in df_7km['Cluster'].unique():
  df_cluster = df_7km[df_7km['Cluster'] == cluster].copy()
  x_lim = (df_cluster['X [m]'].min(), df_cluster['X [m]'].max())
  x_lim = x_lim[0] + (x_lim[1]-x_lim[0])/2
  x_lim = (x_lim-150000, x_lim+150000)
  y_lim = (df_cluster['Y [m]'].min(), df_cluster['Y [m]'].max())
  y_lim = y_lim[0] + (y_lim[1]-y_lim[0])/2
  y_lim = (y_lim-150000, y_lim+150000)

  gdf_cluster = gpd.GeoDataFrame(
    df_cluster, geometry=gpd.points_from_xy(df_cluster.Longitud, df_cluster.Latitud), crs="EPSG:4326"
    )
  gdf_cluster = gdf_cluster.to_crs(9377)

  fig, ax = plt.subplots(figsize=(7, 7), ncols=1, nrows=1)


  gdf_cluster.plot(color='r',
                   ax=ax,
                   zorder=100,
                   label=f'Cluster {cluster} [DBSCAN 7 km]',)

  ax.set_xlim(x_lim[0] , x_lim[1])
  ax.set_ylim(y_lim[0] , y_lim[1])

  cx.add_basemap(ax=ax,
                crs='epsg:9377', # el sistema de coordenadas
                source=xyz.OpenTopoMap,
                reset_extent=True,
                zorder=0,
                zoom=8,
                attribution_size=5) # Para incluir un mapa base

  ax.set_xlabel('X [m]')
  ax.set_ylabel('Y [m]')

  ax.set_aspect('equal')
  plt.grid(True,
          ls='--',
          color='gray',
          alpha=0.8)


  legend  = plt.legend(loc='upper center')
  legend.set_zorder(100)

  sup.plots.save_plot(path_save_figuras = path_save_figures,
                    file_name = f'Map_DBSCAN_7km_cluster_{cluster}',
                    formato = 'png',
                    dpi_ = 300)

  plt.show()
  print('\n')

In [ ]:
for cluster in df_7km['Cluster'].unique():
  fig, ax = plt.subplots(figsize=(10,3), nrows=1, ncols=2)

  filtro_cluster = df_7km['Cluster'] == cluster
  ax[0].hist(df_7km['year'][filtro_cluster],
             bins=np.arange(1992, 2025, 1),
             )
  ax[0].set_xlabel('Year')
  ax[0].set_ylabel('Frequency')
  ax[0].grid(ls='--', alpha= 0.8)

  ax[1].hist(df_7km['Profundidad [km]'][filtro_cluster],
           bins=np.arange(0, 65, 2.5),
           )
  ax[1].set_xlabel('Depth [km]')
  ax[1].set_ylabel('Frequency')
  ax[1].grid(ls='--', alpha= 0.8)

  radios_ = np.logspace(0, 2, 100) # km
  cr = sup.stats.correlacion_integral_xyz(x_km = df_7km['X [km]'][filtro_cluster],
                                        y_km = df_7km['Y [km]'][filtro_cluster],
                                        z_km = df_7km['Profundidad [km]'][filtro_cluster],
                                        radio_en_km = radios_)

  slope, intercepto, otros = sup.stats.estimar_Dc(r_km = radios_,
                                                 C = cr,
                                                 idx_range=None)

  cr_min = radios_[otros[3]][otros[2]][0]
  cr_max = radios_[otros[3]][otros[2]][-1]

  df_centros_7km.loc[df_centros_7km['cluster'] == cluster, 'Cr_min'] = cr_min
  df_centros_7km.loc[df_centros_7km['cluster'] == cluster, 'Cr_max'] = cr_max
  df_centros_7km.loc[df_centros_7km['cluster'] == cluster, 'Dc'] = slope

  plt.suptitle(f'Cluster {cluster} [DBSCAN 7 km] – C(r): {cr_min:.2f}-{cr_max:.2f} km – Dc= {slope:.2f}')
  sup.plots.save_plot(path_save_figuras = path_save_figures,
                    file_name = f'Hist_DBSCAN_7km_cluster_{cluster}',
                    formato = 'png',
                    dpi_ = 300)

  plt.show()
  print()

In [ ]:
dicc_7km = {}

for cluster in df_7km['Cluster'].unique():
  dicc_temp ={'n':[],
              'mc':[],
            'a':[],
            'b':[],
            'mc_std':[],
            'a_std':[],
            'b_std':[],
            'get_N':[]}

  df_cluster = df_7km[df_7km['Cluster'] == cluster].copy()

  dicc_temp['Ml'] = df_cluster['Magnitud'].values

  temp = sup.stats.obtener_N_LGR(magnitudes_array = df_cluster['Magnitud'].values,
                                                  magnitud_minima = mag_min,
                                                  magnitud_maxima = mag_max,
                                                  steps_ = steps_mag,
                                                  texto = False)

  dicc_temp['get_N'].append(temp)

  temp = sup.stats.get_GutenbergRichter_values_boosted_2(magnitudes = df_cluster['Magnitud'],
                                                            percentage_data = porcentaje_remuestreo, # 0.8
                                                            magnitud_minima = mag_min,
                                                            magnitud_maxima = mag_max,
                                                            steps_ = steps_mag,
                                                            n_resamples = numero_remuestreos,
                                                            replace_ = reemplazo,
                                                            correcion_m = correcion_mc)
  dicc_temp['n'].append(df_cluster.shape[0])
  dicc_temp['mc'].append(temp['mc_mean'])
  dicc_temp['a'].append(temp['a_mean'])
  dicc_temp['b'].append(temp['b_mean'])
  dicc_temp['mc_std'].append(temp['mc_std'])
  dicc_temp['a_std'].append(temp['a_std'])
  dicc_temp['b_std'].append(temp['b_std'])

  df_centros_7km.loc[df_centros_7km['cluster'] == cluster, 'n'] = df_cluster.shape[0]
  df_centros_7km.loc[df_centros_7km['cluster'] == cluster, 'mc'] = temp['mc_mean']
  df_centros_7km.loc[df_centros_7km['cluster'] == cluster, 'a'] = temp['a_mean']
  df_centros_7km.loc[df_centros_7km['cluster'] == cluster, 'b'] = temp['b_mean']
  df_centros_7km.loc[df_centros_7km['cluster'] == cluster, 'mc_std'] = temp['mc_std']
  df_centros_7km.loc[df_centros_7km['cluster'] == cluster, 'a_std'] = temp['a_std']
  df_centros_7km.loc[df_centros_7km['cluster'] == cluster, 'b_std'] = temp['b_std']

  dicc_7km[str(cluster)] = dicc_temp

dicc_7km

In [ ]:
for cluster in dicc_7km.keys():
  print(f'\nCluster: {cluster}')
  fig, ax = plt.subplots(nrows=1, ncols=2, figsize=(8,4),
                         sharex='all')

  temp = dicc_7km[cluster]
  min_max_GR = [temp['mc'][0].round(2) - 0.2,
                max(temp['get_N'][0]['m'][temp['get_N'][0]['N'] != 0]) + 0.2]

  sup.plots.histograma_plot(df= pd.DataFrame(temp['Ml'], columns=['Ml']),
                          columna= 'Ml',
                          ax_h= ax[0],
                          dist_min= 0,
                          dist_max= 10,
                          units_label='Ml',
                          step_= 0.2)


  ax[1].axvline(x=3, c='k', ls='--', alpha=0.5)

  sup.plots.GutenbergRichter_scatter_plot(N_count = temp['get_N'][0]['N'],
                                          magnitudes = temp['get_N'][0]['m'],
                                          ax_h = ax[1],
                                          mc = temp['mc'][0].round(2),
                                          magnitud_maxima= mag_max)
  sup.plots.GutenbergRichter_line_plot(b_value = temp['b'][0],
                                        a_value = temp['a'][0],
                                        ax_h = ax[1],
                                        magnitud_minima = min_max_GR[0],
                                        magnitud_maxima = min_max_GR[1])

  ax[1].set_xlabel('Magnitude')
  ax[1].set_title(f'Gutenberg-Richter')

  plt.suptitle(f'Cluster: {cluster} [n={temp['n'][0]}] [DBSCAN 7 km]')
  plt.tight_layout()

  sup.plots.save_plot(path_save_figuras = path_save_figures,
                    file_name = f'LGR_DBSCAN_7km_cluster_{cluster}',
                    formato = 'png',
                    dpi_ = 300)


  plt.show()


# Save CSV

In [ ]:
df_centros_5km.to_csv('centros_DBSCAN_5km.csv',
                     sep=';',
                     decimal=',',
                     index=False)

In [ ]:
df_centros_7km.to_csv('centros_DBSCAN_7km.csv',
                     sep=';',
                     decimal=',',
                     index=False)

# Fin